In [15]:
import pandas as pd
import numpy as np
import pickle
from sklearn.model_selection import train_test_split
import warnings
warnings.filterwarnings('ignore')

df = pd.read_csv(r'D:\VCUBE NOTES\projects\Amazon(RS)\amazon-recsy\data\clean_ratings.csv')

# Load models
with open(r'D:\VCUBE NOTES\projects\Amazon(RS)\amazon-recsy\models\\als_model.pkl', 'rb') as f:
    als_model = pickle.load(f)
with open(r'D:\VCUBE NOTES\projects\Amazon(RS)\amazon-recsy\models\\encoders.pkl', 'rb') as f:
    enc = pickle.load(f)
    user_encoder = enc['user_encoder']
    user_decoder = enc['user_decoder']
    product_encoder = enc['product_encoder']
    product_decoder = enc['product_decoder']
with open(r'D:\VCUBE NOTES\projects\Amazon(RS)\amazon-recsy\models\interaction_matrix.pkl', 'rb') as f:
    interaction_matrix = pickle.load(f)
with open(r'D:\VCUBE NOTES\projects\Amazon(RS)\amazon-recsy\models\combined_matrix.pkl', 'rb') as f:
    combined_matrix = pickle.load(f)
with open(r'D:\VCUBE NOTES\projects\Amazon(RS)\amazon-recsy\models\\product_profiles.pkl', 'rb') as f:
    product_profiles = pickle.load(f)
with open(r'D:\VCUBE NOTES\projects\Amazon(RS)\amazon-recsy\models\\product_idx_map.pkl', 'rb') as f:
    maps = pickle.load(f)
    product_idx_map = maps['product_idx_map']
    product_id_map = maps['product_id_map']

print("✅ All models loaded")
print(f"Total interactions: {len(df):,}")
print(f"Unique users       : {df['user_id'].nunique():,}")
print(f"Unique products    : {df['product_id'].nunique():,}")

✅ All models loaded
Total interactions: 72,696
Unique users       : 10,327
Unique products    : 2,717


In [16]:
# For each user hold out their most recent 20% ratings as test
train_data = []
test_data  = []

for user_id, group in df.groupby('user_id'):
    group     = group.sort_values('timestamp')
    n         = len(group)
    split_idx = max(1, int(n * 0.8))
    train_data.append(group.iloc[:split_idx])
    test_data.append(group.iloc[split_idx:])

train_df = pd.concat(train_data).reset_index(drop=True)
test_df  = pd.concat(test_data).reset_index(drop=True)

# Keep only liked items in test as ground truth
test_df = test_df[test_df['rating'] >= 4]

print(f"Train size   : {len(train_df):,}")
print(f"Test size    : {len(test_df):,}")
print(f"Test users   : {test_df['user_id'].nunique():,}")

Train size   : 54,658
Test size    : 15,844
Test users   : 9,497


In [17]:
def precision_at_k(recommended, relevant, k):
    """Of top-K recommended, how many are actually relevant?"""
    hits = len(set(recommended[:k]) & set(relevant))
    return hits / k if k > 0 else 0

def recall_at_k(recommended, relevant, k):
    """Of all relevant items, how many did we find in top-K?"""
    hits = len(set(recommended[:k]) & set(relevant))
    return hits / len(relevant) if len(relevant) > 0 else 0

def ndcg_at_k(recommended, relevant, k):
    """
    Normalized Discounted Cumulative Gain.
    Rewards finding relevant items early (rank 1 is better than rank 10).
    Score 1.0 = perfect ranking.
    """
    recommended_k = recommended[:k]
    dcg  = sum(
        1 / np.log2(i + 2)
        for i, item in enumerate(recommended_k)
        if item in set(relevant)
    )
    ideal_hits = min(len(relevant), k)
    idcg = sum(1 / np.log2(i + 2) for i in range(ideal_hits))
    return dcg / idcg if idcg > 0 else 0

print("✅ Metrics defined: Precision@K, Recall@K, NDCG@K")

✅ Metrics defined: Precision@K, Recall@K, NDCG@K


In [18]:
# Root cause of ALS = 0.0000:
# Original ALS was trained on full data but evaluated on a split
# Fix: retrain ALS only on train_df so it never sees test data

train_users    = train_df['user_id'].unique()
train_products = train_df['product_id'].unique()

train_user_encoder    = {uid: idx for idx, uid in enumerate(train_users)}
train_user_decoder    = {idx: uid for uid, idx in train_user_encoder.items()}
train_product_encoder = {pid: idx for idx, pid in enumerate(train_products)}
train_product_decoder = {idx: pid for pid, idx in train_product_encoder.items()}

print(f"Train users    : {len(train_user_encoder):,}")
print(f"Train products : {len(train_product_encoder):,}")

Train users    : 10,327
Train products : 2,717


In [27]:
train_df['confidence'] = train_df['rating'].apply(lambda x: 1 + 40 * x)

train_user_idx    = train_df['user_id'].map(train_user_encoder)
train_product_idx = train_df['product_id'].map(train_product_encoder)

# Drop rows where mapping failed
valid_mask        = train_user_idx.notna() & train_product_idx.notna()
train_df_clean    = train_df[valid_mask].copy()
train_user_idx    = train_user_idx[valid_mask].astype(int)
train_product_idx = train_product_idx[valid_mask].astype(int)

train_interaction_matrix = csr_matrix(
    (train_df_clean['confidence'],
     (train_user_idx, train_product_idx)),       # ← swapped
    shape=(len(train_user_encoder),
           len(train_product_encoder))           # ← swapped
)

print(f"Train matrix shape : {train_interaction_matrix.shape}")
print(f"  rows = users     : {len(train_user_encoder):,}")
print(f"  cols = products  : {len(train_product_encoder):,}")
print(f"Interactions       : {train_interaction_matrix.nnz:,}")
sparsity = 1 - (train_interaction_matrix.nnz /
                (len(train_user_encoder) * len(train_product_encoder)))
print(f"Sparsity           : {sparsity:.4%}")

Train matrix shape : (10327, 2717)
  rows = users     : 10,327
  cols = products  : 2,717
Interactions       : 49,497
Sparsity           : 99.8236%


In [28]:
from implicit import als

print("Retraining ALS on train data only...")

train_als_model = als.AlternatingLeastSquares(
    factors=50,
    regularization=0.1,
    iterations=30,
    use_gpu=False
)

# implicit expects (users × items) — now correct
train_als_model.fit(train_interaction_matrix)

print("✅ ALS retrained!")
print(f"  user_factors : {train_als_model.user_factors.shape}")
print(f"  item_factors : {train_als_model.item_factors.shape}")

# Expected output:
# user_factors : (10327, 50)
# item_factors : (2717, 50)

Retraining ALS on train data only...


  0%|          | 0/30 [00:00<?, ?it/s]

✅ ALS retrained!
  user_factors : (10327, 50)
  item_factors : (2717, 50)


In [30]:
def get_als_recs(user_id, n=20):
    if user_id not in train_user_encoder:
        return []

    uid = train_user_encoder[user_id]

    if uid >= train_als_model.user_factors.shape[0]:
        return []

    # Fix: matrix is now (users × products) so no .T needed
    user_items = csr_matrix(train_interaction_matrix[uid])

    ids, scores = train_als_model.recommend(
        uid, user_items,
        N=n,
        filter_already_liked_items=True
    )

    results = []
    for p in ids:
        p = int(p)
        if p in train_product_decoder:
            results.append(train_product_decoder[p])
    return results


def get_content_recs(user_id, n=20):
    liked = train_df[
        (train_df['user_id'] == user_id) &
        (train_df['rating'] >= 4)
    ]
    if liked.empty:
        return []

    acc = np.zeros(len(product_profiles))

    for _, row in liked.iterrows():
        if row['product_id'] not in product_idx_map:
            continue
        idx  = product_idx_map[row['product_id']]
        sims = cosine_similarity(
            combined_matrix[idx], combined_matrix
        ).flatten()
        acc += sims * row['rating']

    rated = set(train_df[train_df['user_id'] == user_id]['product_id'])
    for pid in rated:
        if pid in product_idx_map:
            acc[product_idx_map[pid]] = 0

    top = acc.argsort()[::-1][:n]
    return [product_id_map[i] for i in top if acc[i] > 0]


def get_hybrid_recs(user_id, n=20):
    count = len(train_df[train_df['user_id'] == user_id])

    if count < 5:
        als_w, cb_w = 0.2, 0.8
    elif count < 20:
        als_w, cb_w = 0.5, 0.5
    else:
        als_w, cb_w = 0.8, 0.2

    als_r = get_als_recs(user_id, n=50)
    cb_r  = get_content_recs(user_id, n=50)

    als_scores = {p: (1 - i/len(als_r)) for i, p in enumerate(als_r)} if als_r else {}
    cb_scores  = {p: (1 - i/len(cb_r))  for i, p in enumerate(cb_r)}  if cb_r  else {}

    all_products = set(als_scores.keys()) | set(cb_scores.keys())
    if not all_products:
        return []

    hybrid_scores = {
        pid: als_w * als_scores.get(pid, 0) + cb_w * cb_scores.get(pid, 0)
        for pid in all_products
    }
    return sorted(
        hybrid_scores, key=hybrid_scores.get, reverse=True
    )[:n]


# ── Sanity check ──────────────────────────────────────────
sample_user = train_df['user_id'].iloc[0]
print(f"Sanity check — user: {sample_user}")
print(f"  ALS recs     : {get_als_recs(sample_user,  n=3)}")
print(f"  Content recs : {get_content_recs(sample_user, n=3)}")
print(f"  Hybrid recs  : {get_hybrid_recs(sample_user,  n=3)}")
print("\n✅ All functions ready")


Sanity check — user: A0096681Y127OL1H8W3U
  ALS recs     : ['B0002E2TVM', 'B0002D0DWK', 'B0002CZWXQ']
  Content recs : ['B0002F741Q', 'B0002CZWXQ', 'B002RGPQJ0']
  Hybrid recs  : ['B0002CZWXQ', 'B0002OMOGC', 'B000EENMQG']

✅ All functions ready


In [31]:
eval_users   = test_df['user_id'].unique()
np.random.seed(42)
sample_users = np.random.choice(
    eval_users, size=min(500, len(eval_users)), replace=False
)

results  = {'als': [], 'content': [], 'hybrid': []}
K_VALUES = [5, 10]
skipped  = 0

print(f"Evaluating {len(sample_users)} users...")
print("This may take 2-3 minutes...\n")

for i, user_id in enumerate(sample_users):
    relevant = test_df[
        test_df['user_id'] == user_id
    ]['product_id'].tolist()

    if not relevant:
        skipped += 1
        continue

    try:
        als_recs     = get_als_recs(user_id,     n=10)
        content_recs = get_content_recs(user_id, n=10)
        hybrid_recs  = get_hybrid_recs(user_id,  n=10)
    except Exception:
        skipped += 1
        continue

    for k in K_VALUES:
        for model_name, recs in [('als',     als_recs),
                                  ('content', content_recs),
                                  ('hybrid',  hybrid_recs)]:
            results[model_name].append({
                f'precision@{k}': precision_at_k(recs, relevant, k),
                f'recall@{k}':    recall_at_k(recs, relevant, k),
                f'ndcg@{k}':      ndcg_at_k(recs, relevant, k),
            })

    if (i + 1) % 100 == 0:
        print(f"  Progress: {i+1}/{len(sample_users)} users done...")

evaluated = len(results['hybrid']) // len(K_VALUES)
print(f"\n✅ Evaluation complete!")
print(f"   Evaluated : {evaluated} users")
print(f"   Skipped   : {skipped} users")

Evaluating 500 users...
This may take 2-3 minutes...

  Progress: 100/500 users done...
  Progress: 200/500 users done...
  Progress: 300/500 users done...
  Progress: 400/500 users done...
  Progress: 500/500 users done...

✅ Evaluation complete!
   Evaluated : 500 users
   Skipped   : 0 users


In [32]:
import json, os

print("\n" + "=" * 60)
print("         FINAL MODEL EVALUATION RESULTS")
print("=" * 60)

summary = {}
for model_name, model_results in results.items():
    results_df         = pd.DataFrame(model_results)
    summary[model_name] = results_df.mean()

summary_df = pd.DataFrame(summary).T
summary_df = summary_df[[
    'precision@5', 'recall@5', 'ndcg@5',
    'precision@10', 'recall@10', 'ndcg@10'
]]
summary_df = summary_df.round(4)

print(summary_df.to_string())
print("=" * 60)

print("\n🏆 Best model per metric:")
for col in summary_df.columns:
    best = summary_df[col].idxmax()
    val  = summary_df[col].max()
    print(f"  {col:15} → {best:10} ({val:.4f})")

# Save
os.makedirs('../models', exist_ok=True)
with open('../models/eval_results.json', 'w') as f:
    json.dump(summary_df.to_dict(), f, indent=2)

print("\n✅ Results saved to models/eval_results.json")





         FINAL MODEL EVALUATION RESULTS
         precision@5  recall@5  ndcg@5  precision@10  recall@10  ndcg@10
als           0.0432    0.1390  0.1275        0.0266     0.1716   0.1395
content       0.0448    0.1376  0.1338        0.0288     0.1755   0.1479
hybrid        0.0472    0.1542  0.1407        0.0306     0.1955   0.1560

🏆 Best model per metric:
  precision@5     → hybrid     (0.0472)
  recall@5        → hybrid     (0.1542)
  ndcg@5          → hybrid     (0.1407)
  precision@10    → hybrid     (0.0306)
  recall@10       → hybrid     (0.1955)
  ndcg@10         → hybrid     (0.1560)

✅ Results saved to models/eval_results.json


In [26]:
# ── DIAGNOSIS CELL ──────────────────────────────────────
print("=" * 55)
print("ALS DEBUG DIAGNOSIS")
print("=" * 55)

# Check 1: model shapes
print(f"\n1. Model shapes:")
print(f"   user_factors  : {train_als_model.user_factors.shape}")
print(f"   item_factors  : {train_als_model.item_factors.shape}")
print(f"   train matrix  : {train_interaction_matrix.shape}")

# Check 2: sample user
sample_user = train_df['user_id'].iloc[0]
uid = train_user_encoder[sample_user]
print(f"\n2. Sample user:")
print(f"   user_id : {sample_user}")
print(f"   uid idx : {uid}")
print(f"   in bounds? : {uid < train_als_model.user_factors.shape[0]}")

# Check 3: what does recommend actually return?
user_items = csr_matrix(train_interaction_matrix.T[uid])
ids, scores = train_als_model.recommend(
    uid, user_items, N=10,
    filter_already_liked_items=True
)
print(f"\n3. Raw recommend output:")
print(f"   ids    : {ids}")
print(f"   scores : {scores}")
print(f"   id dtype: {type(ids[0]) if len(ids) > 0 else 'empty'}")

# Check 4: decoder keys
print(f"\n4. Decoder check:")
print(f"   train_product_decoder has {len(train_product_decoder)} keys")
print(f"   First 5 keys: {list(train_product_decoder.keys())[:5]}")
print(f"   Key type: {type(list(train_product_decoder.keys())[0])}")

# Check 5: do ids exist in decoder?
print(f"\n5. ID lookup check:")
for raw_id in ids[:3]:
    as_int = int(raw_id)
    print(f"   raw={raw_id} (type={type(raw_id).__name__}) "
          f"→ int={as_int} "
          f"→ in decoder? {as_int in train_product_decoder}")

# Check 6: does ALS find anything in test users?
print(f"\n6. Test user check:")
test_users_sample = test_df['user_id'].unique()[:5]
for u in test_users_sample:
    in_train = u in train_user_encoder
    print(f"   {u} → in train encoder? {in_train}")

ALS DEBUG DIAGNOSIS

1. Model shapes:
   user_factors  : (2717, 50)
   item_factors  : (10327, 50)
   train matrix  : (2717, 10327)

2. Sample user:
   user_id : A0096681Y127OL1H8W3U
   uid idx : 0
   in bounds? : True

3. Raw recommend output:
   ids    : [ 7593  9206  9718 10320   716  7707  9087  8205  8298  8480]
   scores : [0.86944    0.8508727  0.70973086 0.5748416  0.5429037  0.5360665
 0.53022105 0.5076273  0.5004421  0.4990866 ]
   id dtype: <class 'numpy.int32'>

4. Decoder check:
   train_product_decoder has 2717 keys
   First 5 keys: [0, 1, 2, 3, 4]
   Key type: <class 'int'>

5. ID lookup check:
   raw=7593 (type=int32) → int=7593 → in decoder? False
   raw=9206 (type=int32) → int=9206 → in decoder? False
   raw=9718 (type=int32) → int=9718 → in decoder? False

6. Test user check:
   A0096681Y127OL1H8W3U → in train encoder? True
   A0279100VZXR9A2495P4 → in train encoder? True
   A0955928C2RRWOWZN7UC → in train encoder? True
   A10044ECXDUVKS → in train encoder? True
   A